# Лабораторная работа по теме "Частотный конвертер"
Цель: реализовать частотный конвертер, провести эксперименты с различными типами фильтрации

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import Audio, display
import os
import scipy
from scipy.signal import lfilter
import math

Реализуем частотный конвертер для случая N_1 > N_2:

In [2]:
def frequency_convertor(x, fs_1, fs_2):
    if fs_1 <= fs_2:
        raise ValueError("Эта функция только для уменьшения частоты")
        
    N1 = len(x)
    ratio = fs_2 / fs_1
    N2 = int(N1 * ratio)
    
    if N2 % 2 != 0:
        N2 -= 1
        
    print(f"Исходная длина: {N1}, Новая длина: {N2}")
    
    X = scipy.fft.fft(x)   
    
    half_N2 = N2 // 2
    Y = np.concatenate([X[:half_N2], X[-half_N2:]])
    Y = Y * (N2 / N1)
    y = scipy.fft.ifft(Y)
    
    return np.real(y)

Реализуем частотный конвертер для случая N_1 < N_2:

In [3]:
def frequency_convertor_upscale(x, fs_1, fs_2):
    if fs_1 >= fs_2:
        raise ValueError("Эта функция только для увеличения частоты")
        
    N1 = len(x)
    ratio = fs_2 / fs_1
    N2 = int(N1 * ratio)
    
    if N2 % 2 != 0:
        N2 += 1
        
    print(f"Исходная длина: {N1}, Новая длина: {N2}")
    
    X = scipy.fft.fft(x)
    
    zeros_count = N2 - N1
    half_N1 = N1 // 2
    head = X[:half_N1]
    tail = X[half_N1:] 
    zeros = np.zeros(zeros_count, dtype=complex)
    Y = np.concatenate([head, zeros, tail])
    Y = Y * (N2 / N1)
    
    y = scipy.fft.ifft(Y)
    
    return np.real(y)

Реализуем конвертер с применением фильтра БИХ-фильтра (фильтра Баттерворта)

In [4]:
def convert_with_iir(x, fs_1, fs_2):
    
    gcd = math.gcd(fs_1, fs_2)
    L = fs_2 // gcd
    M = fs_1 // gcd
    
    x_up = np.zeros(len(x) * L)
    x_up[::L] = x
    fs_temp = fs_1 * L
    cutoff = min(fs_1, fs_2) / 2
    b, a = scipy.signal.iirfilter(N=6, Wn=cutoff, fs=fs_temp, btype='lowpass', ftype='butter')
    y_temp = scipy.signal.filtfilt(b, a, x_up)
    y = y_temp[::M]
    
    return y

Реализуем конвертер с применением КИХ - фильтра 

In [5]:
def convert_with_fir(x, fs_1, fs_2, numtaps=201):

    gcd = math.gcd(fs_1, fs_2)
    L = fs_2 // gcd
    M = fs_1 // gcd
    
    x_up = np.zeros(len(x) * L)
    x_up[::L] = x
    fs_temp = fs_1 * L
    cutoff = min(fs_1, fs_2) / 2
    b = scipy.signal.firwin(numtaps, cutoff, fs=fs_temp) * L
    y_temp = scipy.signal.filtfilt(b, 1.0, x_up)
    y = y_temp[::M]
    
    return y

Загружаем тестовый звуковой файл

In [6]:
filename = 'announcement.wav'
y, sr = librosa.load(filename, sr=None)
print(f'Исходная частота {sr} Гц')

Исходная частота 22050 Гц


In [7]:
print("Оригинальный сигнал:")
display(Audio(y, rate=sr))

Оригинальный сигнал:


Реализуем понижение частоты дискретизации с помощью реализованных функий и встроенной функции librosa c 22 050 -> 4 000:

In [8]:
target_sr = 4000 

In [9]:
y_downsampled = frequency_convertor(y, sr, target_sr)

Исходная длина: 155214, Новая длина: 28156


In [10]:
y_downsampled_iir =  convert_with_iir(y, sr, target_sr)

In [11]:
y_downsampled_fir =  convert_with_fir(y, sr, target_sr)

In [12]:
y_downsampled_librosa = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

In [13]:
display(Audio(y_downsampled, rate=target_sr))

In [14]:
display(Audio(y_downsampled_iir, rate=target_sr))

In [15]:
display(Audio(y_downsampled_fir, rate=target_sr))

In [16]:
display(Audio(y_downsampled_librosa, rate=target_sr))

Реализуем повышение частоты дискретизации с помощью реализованных функий и встроенной функции librosa c 22 050 -> 44 100:

In [17]:
target_sr_1 = 44100 

In [18]:
y_upsampled = frequency_convertor_upscale(y, sr, target_sr_1)
y_upsampled_iir = convert_with_iir(y, sr, target_sr_1)
y_upsampled_fir = convert_with_fir(y, sr, target_sr_1)
y_upsampled_librosa = librosa.resample(y, orig_sr=sr, target_sr=target_sr_1)


Исходная длина: 155214, Новая длина: 310428


In [19]:
display(Audio(y_upsampled, rate=target_sr_1))
display(Audio(y_upsampled_iir, rate=target_sr_1))
display(Audio(y_upsampled_fir, rate=target_sr_1))
display(Audio(y_upsampled_librosa, rate=target_sr_1))

### Выводы 

В результате выполнения работы мы реализовали частотный конвертер для понижения и повышения частоты дискретизации.

При применении фильтров для понижения частоты дискретизации с 22 050 до 4 000 гораздо сильнее слышатся шумы, качество хуже в сравнении со встроенной библиотекой librosa.
При повышении частоты дискретизации с 22 050 до 44 100 существенных различий не выявлено.